# 01 Iron anode (IA-FE-001)

**Requirement:** `IA-FE-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

$$
\newcommand{\per}{/}
\newcommand{\squared}{^{2}}
\newcommand{\cubed}{^{3}}
\newcommand{\of}[1]{\,\mathrm{#1}}
\newcommand{\milli}{\mathrm{m}}
\newcommand{\centi}{\mathrm{c}}
\newcommand{\micro}{\mathrm{\mu}}
\newcommand{\kilo}{\mathrm{k}}
\newcommand{\metre}{\mathrm{m}}
\newcommand{\gram}{\mathrm{g}}
\newcommand{\kilogram}{\mathrm{kg}}
\newcommand{\second}{\mathrm{s}}
\newcommand{\minute}{\mathrm{min}}
\newcommand{\hour}{\mathrm{h}}
\newcommand{\day}{\mathrm{d}}
\newcommand{\ampere}{\mathrm{A}}
\newcommand{\volt}{\mathrm{V}}
\newcommand{\watt}{\mathrm{W}}
\newcommand{\ohm}{\mathrm{\Omega}}
\newcommand{\kelvin}{\mathrm{K}}
\newcommand{\celsius}{{}^{\circ}\mathrm{C}}
\newcommand{\coulomb}{\mathrm{C}}
\newcommand{\joule}{\mathrm{J}}
\newcommand{\pascal}{\mathrm{Pa}}
\newcommand{\litre}{\mathrm{L}}
\newcommand{\mole}{\mathrm{mol}}
\newcommand{\molar}{\mathrm{M}}
\newcommand{\percent}{\%}
\newcommand{\atm}{\mathrm{atm}}
\newcommand{\newton}{\mathrm{N}}
\newcommand{\qty}[2]{#1\,#2}
\newcommand{\qtyrange}[3]{#1\text{--}#2\,#3}
\newcommand{\unit}[1]{#1}
\newcommand{\si}[1]{#1}
$$


## Governing equations

Discharge of metallic iron in alkaline electrolyte (EP4602674A1 [0042]-[0043] Eq. 1),
theoretical capacity $\qty{960}{\milli\ampere\hour\per\gram}$ Fe:

$$\mathrm{Fe} + 2\,\mathrm{OH^-} \leftrightarrow \mathrm{Fe(OH)_2} + 2\,\mathrm{e^-}$$

Second discharge step (Eq. 2), $\qty{320}{\milli\ampere\hour\per\gram}$ Fe:

$$3\,\mathrm{Fe(OH)_2} + 2\,\mathrm{OH^-} \leftrightarrow \mathrm{Fe_3O_4} + 4\,\mathrm{H_2O} + 2\,\mathrm{e^-}$$

Faraday: $r = I/(nF)$, $n=2$. Butler-Volmer current from `ironair.chemistry.kinetics`.
Porosity absorbs the solid volume increase (IA-SYS-020). DRI beds, porous particles, and
channeled ribs (EP Claim 1; loading $\qtyrange{1}{7}{\gram\per\centi\metre\squared}$, VF $0.5$–$0.9$) share this RHS.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params
from plant_sim.siunitx_nb import SIUNITX_PREAMBLE

In [ ]:
from IPython import get_ipython
from IPython.display import Markdown, display

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
display(Markdown(SIUNITX_PREAMBLE))
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Isolated component simulation

In [ ]:
from plant_sim.components.iron_anode import IronAnode
from plant_sim.components.base import integrate_component

fe = IronAnode(P)
I = 40 * P.I_discharge_100h_A
t, y = integrate_component(
    lambda t, y: fe.rhs(t, y, {"I_fe_A": I, "T_amb_K": P.T_ep_sim_K, "a_oh": 6.0, "a_h2o": 0.72}, {}),
    fe.y0(),
    (0, 4000),
    n_eval=200,
)
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "n_Fe": (y[0]),
        "n_FeOH2": (y[1]),
        "n_Fe3O4": (y[2]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("mol")
ax.set_title("Iron phase inventories")
ax.text(0.0, -0.22, "EP [0042]-[0043]", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
cap = [
    fe.outputs(float(ti), y[:, i], {"I_fe_A": I, "a_oh": 6.0, "a_h2o": 0.72}, {})["capacity_extracted_mAh_g"]
    for i, ti in enumerate(t)
]
print("extracted mA h g⁻¹ (accelerated)", cap[-1], "theory 960 =", CAPACITY_FE_OH2_MAH_G)

## Verification against patents / SSS002

In [ ]:
assert abs(CAPACITY_FE_OH2_MAH_G - 960) < 1.0
assert abs(CAPACITY_MAGNETITE_MAH_G - 320) < 1.0
print("PASS Faraday capacities")
print("loading", P.loading_g_cm2, "g cm⁻² in [1, 7]; VF", P.vf_electrolyte_charged, "in [0.5, 0.9]")